### Instalar librerías, ya creado el entorno y verificado que se puede ejecutar el script de Python

In [12]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import sklearn as sk


### Importar archivo remuneraciones extención XLSX, usando como encabezado la primera fila útil

In [15]:
archivos = r"D:\Stella\Documentos\POLITECNICO\2 año\Aprendizaje automático\Insumos Parcial\Insumos para el modelo\21_1_01_Remuneracion-promedio-de-los-trabajadores-registrados-del-sector-privado-segun-rama-de-actividad.xlsx"

df_remuneraciones = pd.read_excel(archivos, engine="openpyxl", header=2,)
print(df_remuneraciones.head())

  Rama de Actividad                                       Unnamed: 1  \
0                 A      AGRICULTURA, GANADERIA, CAZA Y SILVICULTURA   
1                 1                         Agricultura y ganadería    
2                 2              Silvicultura, extracción de madera    
3                 B                        PESCA Y SERVICIOS CONEXOS   
4                 5   Pesca y actividades relacionadas con la pesca    

  1995-01-01 00:00:00 1995-02-01 00:00:00 1995-03-01 00:00:00  \
0                 797                 738                 853   
1                 923                 821                 974   
2                 462                 517                 552   
3                1282                1229                1360   
4                1282                1229                1360   

  1995-04-01 00:00:00 1995-05-01 00:00:00 1995-06-01 00:00:00  \
0                 527                 469                1049   
1                 528                 461     

### Seleccionar las columnas útiles

In [21]:
print(df_remuneraciones.columns.tolist())
print(type(df_remuneraciones.columns))

['Rama de Actividad', 'Unnamed: 1', datetime.datetime(1995, 1, 1, 0, 0), datetime.datetime(1995, 2, 1, 0, 0), datetime.datetime(1995, 3, 1, 0, 0), datetime.datetime(1995, 4, 1, 0, 0), datetime.datetime(1995, 5, 1, 0, 0), datetime.datetime(1995, 6, 1, 0, 0), datetime.datetime(1995, 7, 1, 0, 0), datetime.datetime(1995, 8, 1, 0, 0), datetime.datetime(1995, 9, 1, 0, 0), datetime.datetime(1995, 10, 1, 0, 0), datetime.datetime(1995, 11, 1, 0, 0), datetime.datetime(1995, 12, 1, 0, 0), datetime.datetime(1996, 1, 1, 0, 0), datetime.datetime(1996, 2, 1, 0, 0), datetime.datetime(1996, 3, 1, 0, 0), datetime.datetime(1996, 4, 1, 0, 0), datetime.datetime(1996, 5, 1, 0, 0), datetime.datetime(1996, 6, 1, 0, 0), datetime.datetime(1996, 7, 1, 0, 0), datetime.datetime(1996, 8, 1, 0, 0), datetime.datetime(1996, 9, 1, 0, 0), datetime.datetime(1996, 10, 1, 0, 0), datetime.datetime(1996, 11, 1, 0, 0), datetime.datetime(1996, 12, 1, 0, 0), datetime.datetime(1997, 1, 1, 0, 0), datetime.datetime(1997, 2, 1, 0, 

In [22]:
# Cambiar el nombre de las columnas para que sean más descriptivas o fáciles de usar
# Por ejemplo, renombrar las dos primeras columnas y dejar las fechas igual

df_remuneraciones = df_remuneraciones.rename(columns={
    df_remuneraciones.columns[0]: "Codigo",
    df_remuneraciones.columns[1]: "Rama_Actividad"
})

print(df_remuneraciones.columns[:10])  # Mostrar los primeros 10 nombres de columnas para verificar

Index([           'Codigo',    'Rama_Actividad', 1995-01-01 00:00:00,
       1995-02-01 00:00:00, 1995-03-01 00:00:00, 1995-04-01 00:00:00,
       1995-05-01 00:00:00, 1995-06-01 00:00:00, 1995-07-01 00:00:00,
       1995-08-01 00:00:00],
      dtype='object')


### Renombrar las columnas, para poder luego seleccionar aquellas que formen parte del análisis y posterior entrenamiento del modelo

In [ ]:
# Renombrar las columnas desde "Rama_Actividad" en adelante para que tengan formato de fecha YYYY-MM
# Primero identificamos el índice de la columna "Rama_Actividad"
idx = df_remuneraciones.columns.get_loc("Rama_Actividad")

# Las columnas de fechas empiezan en el siguiente índice
fecha_cols = df_remuneraciones.columns[(idx+1):]

# Creamos nuevos nombres en formato YYYY-MM
nuevos_nombres = {}
for col in fecha_cols:
    try:
        # Intentar convertir el nombre de la columna a formato fecha
        fecha = pd.to_datetime(col, format='%Y-%m')
        nuevos_nombres[col] = fecha.strftime('%Y-%m')
    except Exception:
        # Si no se puede convertir, dejar el nombre original
        nuevos_nombres[col] = col

# Renombrar las columnas en el DataFrame
df_remuneraciones = df_remuneraciones.rename(columns=nuevos_nombres)

print(df_remuneraciones.columns[idx:idx+10])  # Mostrar los primeros 10 nombres después de "Rama_Actividad"

In [26]:
# Seleccionar las columnas "Codigo", "Rama_Actividad" y desde "2017-12" hasta el final
cols = df_remuneraciones.columns.tolist()
start_idx = cols.index("2017-12")
selected_cols = ["Codigo", "Rama_Actividad"] + cols[start_idx:]
df_remuneraciones_sel = df_remuneraciones[selected_cols]
print(df_remuneraciones_sel.head())

  Codigo                                   Rama_Actividad 2017-12 2018-01  \
0      A      AGRICULTURA, GANADERIA, CAZA Y SILVICULTURA   29477   20495   
1      1                         Agricultura y ganadería    30861   21524   
2      2              Silvicultura, extracción de madera    22224   14432   
3      B                        PESCA Y SERVICIOS CONEXOS   72125   56527   
4      5   Pesca y actividades relacionadas con la pesca    72125   56527   

  2018-02 2018-03 2018-04 2018-05 2018-06 2018-07  ...  2024-03  2024-04  \
0   20629   22372   22193   23158   33354   23985  ...   678022   794914   
1   22011   24093   24166   25689   36915   26425  ...   760692   905057   
2   13803   15061   13702   13125   20094   13129  ...   444725   454192   
3   53876   51386   56918   62512   99129   57314  ...  2503400  2329619   
4   53876   51386   56918   62512   99129   57314  ...  2503400  2329619   

   2024-05  2024-06  2024-07  2024-08  2024-09  2024-10  2024-11  2024-12  
0   

### Hasta acá parece que el primer DB está ok, ahora podría ver algún gráfico de correlación para explorar datos
*podría ver si hay registros nulos o duplicados*

In [27]:
# Verificar registros nulos
print("Cantidad de valores nulos por columna:")
print(df_remuneraciones_sel.isnull().sum())

# Verificar registros duplicados
duplicados = df_remuneraciones_sel.duplicated()
print(f"\nCantidad de filas duplicadas: {duplicados.sum()}")

Cantidad de valores nulos por columna:
Codigo            0
Rama_Actividad    3
2017-12           1
2018-01           1
2018-02           1
                 ..
2024-08           1
2024-09           1
2024-10           1
2024-11           1
2024-12           1
Length: 87, dtype: int64

Cantidad de filas duplicadas: 0
